# Corner-WFS Donut Selection Stages & Interactive Pair / Wavefront Inspector

**Author:** Aaron Roodman
**Date Created:** 2026-06-13
**Last Modified:** 2026-06-14
**Status:** In Progress
**Keywords:** WFS, corner wavefront sensor, donut selection, ts_wep, danish, zernikes, interactive

## Description

For each corner-WFS half-chip (intra / extra), show the **post-ISR image** with
circles overlaid on the donuts, coloured by ts_wep pipeline stage
(selected → fit → used). Then an **interactive pair inspector**: both halves of one
corner raft side by side — click a donut in either panel and it

1. highlights the clicked donut **and its paired donut** in the other panel, and
2. re-runs the **joint Danish fit** of the pair (`danish.DZMultiDonutModel` via
   `WfEstimator`, `jointFitPair=True`, `saveHistory=True`) and shows the
   **data / model / residual** postage stamps for both intra and extra.

The forward-model image is not persisted by the pipeline, so it is regenerated on the
fly from the saved donut stamps (the fit takes ~1 s/pair).

Stage definitions (join key `donut_id`):
- **selected** — in `donutTable` (round-1 direct detection).
- **fit** — member of a pair in `aggregateAOSVisitTableRaw`.
- **used** — member of a pair with `used == True`.

See `ts_wep_cwfs_dataflow.md` for the full product flow.

**Output:** a per-visit PDF of image+circle overlays, an extended `aggregateDonutTable`
(ECSV), and the interactive inspector.

**Based on:** AOS `cwfs` reductions in
`LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2` (repo `/repo/main`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-13 | Aaron Roodman | Initial donut selection-stages notebook (per-CCD selected/fit/used + aggregate table) |
| 2026-06-13 | Aaron Roodman | Overlay stage circles on the CWFS images; add interactive click-to-inspect tool |
| 2026-06-14 | Aaron Roodman | Pair inspector: both intra+extra panels, click→highlight paired donut, joint Danish refit with data/model/residual stamps |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Select Visits](#select)
6. [Image + Stage-Circle Overlays](#plots)
7. [Aggregate Table (fit donuts + used flag)](#table)
8. [Interactive Pair / Wavefront Inspector](#interactive)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
collection = "LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2"
dataset_type = "post_isr_image"
instrument = "LSSTCam"
cam_name = "LSSTCam"

day_obs = 20260327               # night to inspect
max_visits = 3                   # how many visits to render (None = all on the night)
start_index = 0

# Corner wavefront-sensor half-chips. SW0 = extra-focal, SW1 = intra-focal;
# the paired fit products (zernikes / aggregateAOSVisitTableRaw / donutStamps*)
# live on the SW0 / extra detector.
WFS_DETECTORS = {
    191: "R00_SW0", 192: "R00_SW1",
    195: "R04_SW0", 196: "R04_SW1",
    199: "R40_SW0", 200: "R40_SW1",
    203: "R44_SW0", 204: "R44_SW1",
}
DEFOCAL = {"SW0": "extra", "SW1": "intra"}

# ---- Circle overlay ----------------------------------------------------
# Typical donuts are ~150 px across; draw circles a bit larger. Open circles
# (no fill); stage shown by colour only.
donut_diam_px = 150
circle_radius_px = 90
circle_lw = 1.5
color_selected = "deepskyblue"   # detected, not fit
color_fit = "orange"             # fit but not used
color_used = "lime"              # used in the averaged estimate

# ---- Image stretch (sky-tuned) -----------------------------------------
sky_lo_sigma = 2.0
sky_hi_sigma = 8.0
asinh_a = 0.1
cmap = "gray"

# ---- Interactive pair inspector ----------------------------------------
inspect_visit = None             # None -> first rendered visit
inspect_raft = "R00"             # which corner raft (shows its SW0 + SW1)
select_tol_px = 120              # click must be within this many px of a centroid
danish_algo = "danish"           # WfEstimator algorithm
danish_jointFitPair = True       # fit intra+extra together (DZMultiDonutModel)
# Noll indices to refit. None -> use this collection's set (read from the saved
# `zernikes` table); mirrors EstimateZernikesDanishTask.
noll_indices = None

# ---- Layout / output ---------------------------------------------------
output_dir = "output"
output_pdf = None                # None -> output/wfs_donut_stages_{day_obs}.pdf
save_table = True
panel_width = 5.0
title_fontsize = 9
dpi = 150


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages
from astropy.stats import sigma_clipped_stats
from astropy.table import vstack
from astropy.visualization import AsinhStretch, ImageNormalize

import danish
import lsst.daf.butler as dafButler
from lsst.ts.wep.estimation import WfEstimator
from lsst.ts.wep.utils import getTaskInstrument

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()
os.makedirs(output_dir, exist_ok=True)

# The joint-fit model/residual stamps use whatever danish the kernel has.
# (CLI d_latest stack = 1.0.0; the RSP notebook image may be newer, e.g. 1.1.1.)
print("danish version:", danish.__version__)


<a id='functions'></a>
## Helper Functions

In [ ]:
def list_visits(butler, collection, instrument, day_obs):
    """Visits on a night that were fit (have an aggregateAOSVisitTableRaw)."""
    refs = butler.query_datasets(
        "aggregateAOSVisitTableRaw", collections=collection,
        where=f"instrument='{instrument}' and exposure.day_obs={day_obs}",
        explain=False, limit=None)
    return sorted({r.dataId["visit"] for r in refs})


def load_aos_visit_table(butler, collection, instrument, visit):
    """Per-visit fit-pair table (one row per fit pair, with `used`)."""
    return butler.get("aggregateAOSVisitTableRaw", collections=collection,
                      dataId={"instrument": instrument, "visit": visit})


def stage_sets(aos_table):
    """(fit_ids, used_ids) sets of donut_id from an aggregateAOSVisitTableRaw."""
    intra = np.asarray(aos_table["donut_id_intra"])
    extra = np.asarray(aos_table["donut_id_extra"])
    used = np.asarray(aos_table["used"]).astype(bool)
    return (set(intra) | set(extra)), (set(intra[used]) | set(extra[used]))


def load_image(butler, collection, dataset_type, instrument, visit, detector):
    """Post-ISR image array (electrons) for one half-chip.

    `post_isr_image` is keyed by `exposure` (donut tables use `visit`); for
    these single-snap CWFS visits the two ids are equal.
    """
    exp = butler.get(dataset_type, collections=collection,
                     dataId={"instrument": instrument, "exposure": visit,
                             "detector": detector})
    return np.asarray(exp.image.array, dtype=float)


def load_donut_table(butler, collection, instrument, visit, detector):
    """Round-1 selected donuts for one detector, or None."""
    try:
        return butler.get("donutTable", collections=collection,
                          dataId={"instrument": instrument, "visit": visit,
                                  "detector": detector})
    except Exception:
        return None


def load_stamps(butler, collection, instrument, visit, sw0_detector):
    """Donut stamps for a raft (both extra and intra live on the SW0 detector)."""
    did = {"instrument": instrument, "visit": visit, "detector": sw0_detector}
    return (butler.get("donutStampsExtra", collections=collection, dataId=did),
            butler.get("donutStampsIntra", collections=collection, dataId=did))


def find_stamp(stamps, donut_id):
    """Return the DonutStamp with the given donut_id, or None."""
    for s in stamps:
        if s.donut_id == donut_id:
            return s
    return None


def get_noll_indices(butler, collection, instrument, visit, detector):
    """Noll indices this collection fit, read from the saved `zernikes` table.

    Returns the sorted Z-column indices (e.g. [4..19, 22..26] for wep_17_3_0), so
    the interactive refit fits the same Zernikes the pipeline did. Falls back to
    Noll 4-28 (the ts_wep default) if the table can't be read.
    """
    try:
        z = butler.get("zernikes", collections=collection,
                       dataId={"instrument": instrument, "visit": visit,
                               "detector": detector})
        js = sorted(int(c[1:]) for c in z.colnames if re.fullmatch(r"Z\d+", c))
        return js or list(range(4, 29))
    except Exception:
        return list(range(4, 29))


def sky_norm(data, lo_sigma=2.0, hi_sigma=8.0, asinh_a=0.1):
    """ImageNormalize (asinh) centred on the sigma-clipped sky level."""
    finite = data[np.isfinite(data)]
    _, med, std = sigma_clipped_stats(finite, sigma=3.0, maxiters=5)
    return ImageNormalize(vmin=med - lo_sigma * std, vmax=med + hi_sigma * std,
                          stretch=AsinhStretch(a=asinh_a))


def stage_of(donut_id, fit_ids, used_ids):
    """'used' | 'fit' | 'selected' for one donut_id."""
    if donut_id in used_ids:
        return "used"
    if donut_id in fit_ids:
        return "fit"
    return "selected"


def overlay_circles(ax, donut_table, fit_ids, used_ids, *, radius, lw,
                    colors, counts=None):
    """Draw an open circle on each donut, coloured by stage."""
    if donut_table is None:
        return
    ids = np.asarray(donut_table["donut_id"])
    x = np.asarray(donut_table["centroid_x"]); y = np.asarray(donut_table["centroid_y"])
    for i, did in enumerate(ids):
        st = stage_of(did, fit_ids, used_ids)
        if counts is not None:
            counts[st] += 1
        ax.add_patch(Circle((x[i], y[i]), radius, fill=False, ec=colors[st],
                            lw=lw, alpha=0.9))


def pair_row_for(aos_table, donut_id):
    """Row of aggregateAOSVisitTableRaw whose pair contains donut_id, else None."""
    intra = np.asarray(aos_table["donut_id_intra"])
    extra = np.asarray(aos_table["donut_id_extra"])
    idx = np.where((intra == donut_id) | (extra == donut_id))[0]
    return aos_table[int(idx[0])] if len(idx) else None


def joint_danish_fit(extra_stamp, intra_stamp, det_name, noll_indices, *,
                     cam_name="LSSTCam", algo="danish", jointFitPair=True):
    """Re-run the joint Danish fit of a pair; return data/model/residual stamps.

    Mirrors EstimateZernikesDanishTask: WfEstimator with the same nollIndices,
    startWithIntrinsic=True, units="um". Returns dict
    {"extra": (data, model, resid), "intra": (...), "zk": array (um)}.
    The pipeline does not persist the forward-model image, so we regenerate it
    here via saveHistory=True (~1 s/pair).
    """
    inst = getTaskInstrument(cam_name, det_name)
    wf = WfEstimator(algoName=algo, algoConfig={"jointFitPair": jointFitPair},
                     instConfig=inst, nollIndices=list(noll_indices),
                     startWithIntrinsic=True, units="um", saveHistory=True)
    zk, _ = wf.estimateZk(extra_stamp.wep_im, intra_stamp.wep_im)
    h = wf.history
    out = {"zk": np.asarray(zk)}
    for foc in ("extra", "intra"):
        img = np.asarray(h[foc]["image"]); mod = np.asarray(h[foc]["model"])
        out[foc] = (img, mod, img - mod)
    return out


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo, collections=collection)
print("Repo:", butler_repo, "| Collection:", collection)


<a id='select'></a>
## Select Visits

In [ ]:
all_visits = list_visits(butler, collection, instrument, day_obs)
print(f"day_obs {day_obs}: {len(all_visits)} fit visits")
visits = all_visits[start_index:]
if max_visits is not None:
    visits = visits[:max_visits]
print("Rendering visits:", visits)


<a id='plots'></a>
## Image + Stage-Circle Overlays

In [ ]:
rafts = sorted({n.split("_")[0] for n in WFS_DETECTORS.values()})
name_to_det = {v: k for k, v in WFS_DETECTORS.items()}
STAGE_COLORS = {"selected": color_selected, "fit": color_fit, "used": color_used}

legend_handles = [
    Line2D([0], [0], marker="o", mfc="none", mec=color_selected, ms=10, mew=1.6,
           ls="none", label="selected (round 1)"),
    Line2D([0], [0], marker="o", mfc="none", mec=color_fit, ms=10, mew=1.6,
           ls="none", label="fit (in a pair)"),
    Line2D([0], [0], marker="o", mfc="none", mec=color_used, ms=10, mew=1.6,
           ls="none", label="used (used pair)"),
]


def plot_visit_overlay(visit, aos_table):
    """4x2 grid of CWFS images with stage circles overlaid."""
    fit_ids, used_ids = stage_sets(aos_table)
    panel_h = panel_width * 2000.0 / 4072.0
    fig, axes = plt.subplots(len(rafts), 2,
                             figsize=(2 * panel_width, len(rafts) * panel_h),
                             constrained_layout=True)
    axes = np.atleast_2d(axes)
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, wspace=0.01, hspace=0.05)
    for i, raft in enumerate(rafts):
        for j, sw in enumerate(("SW0", "SW1")):
            ax = axes[i, j]; name = f"{raft}_{sw}"; det = name_to_det[name]
            img = load_image(butler, collection, dataset_type, instrument, visit, det)
            ax.imshow(img, origin="lower", cmap=cmap, norm=sky_norm(
                img, sky_lo_sigma, sky_hi_sigma, asinh_a),
                aspect="equal", interpolation="nearest")
            dt = load_donut_table(butler, collection, instrument, visit, det)
            counts = dict(selected=0, fit=0, used=0)
            overlay_circles(ax, dt, fit_ids, used_ids, radius=circle_radius_px,
                            lw=circle_lw, colors=STAGE_COLORS, counts=counts)
            nsel = counts["selected"] + counts["fit"] + counts["used"]
            ax.set_title(f"{name} ({det}, {DEFOCAL[sw]})  sel={nsel} "
                         f"fit={counts['fit']+counts['used']} used={counts['used']}",
                         fontsize=title_fontsize)
            ax.set_xticks([]); ax.set_yticks([])
    fig.legend(handles=legend_handles, loc="upper right", fontsize=8, ncol=3)
    fig.suptitle(f"Visit {visit} — corner-WFS donut selection stages",
                 fontsize=title_fontsize + 3)
    return fig


pdf_path = Path(output_pdf) if output_pdf else \
    Path(output_dir) / f"wfs_donut_stages_{day_obs}.pdf"

agg_tables = []
with PdfPages(pdf_path) as pdf:
    for visit in visits:
        aos = load_aos_visit_table(butler, collection, instrument, visit)
        fig = plot_visit_overlay(visit, aos)
        pdf.savefig(fig, dpi=dpi)
        plt.show()
        fit_ids, used_ids = stage_sets(aos)
        agg = butler.get("aggregateDonutTable", collections=collection,
                         dataId={"instrument": instrument, "visit": visit})
        ids = np.asarray(agg["donut_id"])
        agg["visit"] = visit
        agg["fit"] = np.array([d in fit_ids for d in ids])
        agg["used"] = np.array([d in used_ids for d in ids])
        agg_tables.append(agg)
        print(f"visit {visit}: fit donuts={len(fit_ids)}, used donuts={len(used_ids)}")

print(f"\nWrote {len(visits)} page(s) to {pdf_path}")


<a id='table'></a>
## Aggregate Table (fit donuts + used flag)

In [ ]:
aggregate = vstack(agg_tables, metadata_conflicts="silent") if agg_tables else None
if aggregate is not None:
    print(f"aggregate rows: {len(aggregate)}  "
          f"(fit={int(np.sum(aggregate['fit']))}, used={int(np.sum(aggregate['used']))})")
    fit_donuts = aggregate[aggregate["fit"]]
    cols = [c for c in ["visit", "detector", "donut_id", "centroid_x", "centroid_y",
                        "thx_OCS", "thy_OCS", "snr", "fit", "used"]
            if c in aggregate.colnames]
    print(fit_donuts[cols][:12])
    if save_table:
        out = Path(output_dir) / f"wfs_donut_aggregate_{day_obs}.ecsv"
        aggregate.write(out, overwrite=True)
        print("\nWrote table:", out)


<a id='interactive'></a>
## Interactive Pair / Wavefront Inspector

Both halves of one corner raft are shown side by side (SW0 = extra, SW1 = intra), with
the stage circles overlaid. **Click a donut in either panel**: it highlights the clicked
donut and its paired donut in the other panel, then re-runs the joint Danish fit and
shows the **data / model / residual** stamps for both. Clicking an unpaired (not-fit)
donut shows just its data stamp.

Requires the interactive backend (`ipympl`); run the `%matplotlib widget` cell first.
Each fit takes ~1 s and is cached per pair.

In [ ]:
%matplotlib widget

In [ ]:
class DonutPairInspector:
    """Two-panel (extra+intra) corner-raft inspector with joint-fit stamps."""

    def __init__(self, visit, raft):
        self.visit = visit
        self.raft = raft
        self.det = {"extra": name_to_det[f"{raft}_SW0"],
                    "intra": name_to_det[f"{raft}_SW1"]}
        self.det_name_extra = f"{raft}_SW0"

        self.img, self.xyid = {}, {}
        for foc in ("extra", "intra"):
            self.img[foc] = load_image(butler, collection, dataset_type, instrument,
                                       visit, self.det[foc])
            dt = load_donut_table(butler, collection, instrument, visit, self.det[foc])
            self.xyid[foc] = (np.asarray(dt["centroid_x"]),
                              np.asarray(dt["centroid_y"]),
                              np.asarray(dt["donut_id"]))
        self.aos = load_aos_visit_table(butler, collection, instrument, visit)
        self.fit_ids, self.used_ids = stage_sets(self.aos)
        self.stampsE, self.stampsI = load_stamps(butler, collection, instrument,
                                                 visit, self.det["extra"])
        # Noll indices to refit: use this collection's set (or the `noll_indices`
        # parameter override) so the model matches the production fit.
        self.noll = (noll_indices if noll_indices is not None else
                     get_noll_indices(butler, collection, instrument, visit,
                                      self.det["extra"]))
        self._cache = {}

        # Layout: two image panels on top, 2x3 stamp grid below.
        self.fig = plt.figure(figsize=(13, 11))
        gs = self.fig.add_gridspec(3, 6, height_ratios=[2.2, 1, 1],
                                   hspace=0.28, wspace=0.35)
        self.ax_img = {"extra": self.fig.add_subplot(gs[0, 0:3]),
                       "intra": self.fig.add_subplot(gs[0, 3:6])}
        self.ax_st = {}
        for r, foc in enumerate(("extra", "intra")):
            for c, kind in enumerate(("data", "model", "residual")):
                self.ax_st[(foc, kind)] = self.fig.add_subplot(gs[r + 1, 2*c:2*c+2])
        self.marker = {}
        for foc in ("extra", "intra"):
            ax = self.ax_img[foc]
            ax.imshow(self.img[foc], origin="lower", cmap=cmap,
                      norm=sky_norm(self.img[foc], sky_lo_sigma, sky_hi_sigma, asinh_a),
                      aspect="equal", interpolation="nearest")
            self._circles(ax, foc)
            ax.set_title(f"{raft}_{'SW0' if foc=='extra' else 'SW1'} "
                         f"({self.det[foc]}, {foc}) — click a donut",
                         fontsize=title_fontsize)
            ax.set_xticks([]); ax.set_yticks([])
            (self.marker[foc],) = ax.plot([], [], "x", color="red", ms=13, mew=2.5)
        for ax in self.ax_st.values():
            ax.set_xticks([]); ax.set_yticks([])
        self.fig.suptitle(f"Visit {visit} — raft {raft} donut pair inspector "
                          f"(Noll {self.noll[0]}-{self.noll[-1]})",
                          fontsize=title_fontsize + 3)
        self.cid = self.fig.canvas.mpl_connect("button_press_event", self.on_click)

    def _circles(self, ax, foc):
        x, y, ids = self.xyid[foc]
        for i, did in enumerate(ids):
            ax.add_patch(Circle((x[i], y[i]), circle_radius_px, fill=False,
                                ec=STAGE_COLORS[stage_of(did, self.fit_ids, self.used_ids)],
                                lw=circle_lw, alpha=0.9))

    def _fit(self, eid, iid):
        if (eid, iid) not in self._cache:
            es = find_stamp(self.stampsE, eid)
            is_ = find_stamp(self.stampsI, iid)
            self._cache[(eid, iid)] = joint_danish_fit(
                es, is_, self.det_name_extra, self.noll, cam_name=cam_name,
                algo=danish_algo, jointFitPair=danish_jointFitPair)
        return self._cache[(eid, iid)]

    def _show_stamp(self, foc, kind, arr):
        ax = self.ax_st[(foc, kind)]; ax.clear()
        if kind == "residual":
            v = np.nanpercentile(np.abs(arr), 99)
            ax.imshow(arr, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
        else:
            ax.imshow(arr, origin="lower", cmap="viridis")
        ax.set_title(f"{foc} {kind}", fontsize=title_fontsize - 1)
        ax.set_xticks([]); ax.set_yticks([])

    def _clear_stamps(self):
        for (foc, kind), ax in self.ax_st.items():
            ax.clear(); ax.set_title(f"{foc} {kind}", fontsize=title_fontsize - 1)
            ax.set_xticks([]); ax.set_yticks([])

    def on_click(self, event):
        foc = next((f for f in ("extra", "intra")
                    if event.inaxes is self.ax_img[f]), None)
        if foc is None or event.xdata is None:
            return
        x, y, ids = self.xyid[foc]
        d = np.hypot(x - event.xdata, y - event.ydata)
        i = int(np.argmin(d))
        if d[i] > select_tol_px:
            print(f"no donut within {select_tol_px} px")
            return
        did = ids[i]
        self.marker[foc].set_data([x[i]], [y[i]])
        other = "intra" if foc == "extra" else "extra"
        row = pair_row_for(self.aos, did)
        self._clear_stamps()
        if row is None:
            self.marker[other].set_data([], [])
            st = stage_of(did, self.fit_ids, self.used_ids)
            stamp = find_stamp(self.stampsE if foc == "extra" else self.stampsI, did)
            if stamp is not None:
                self._show_stamp(foc, "data", np.asarray(stamp.wep_im.image))
            print(f"{did}: stage={st} — not in a fit pair")
        else:
            eid, iid = row["donut_id_extra"], row["donut_id_intra"]
            pid = iid if foc == "extra" else eid
            ox, oy, oids = self.xyid[other]
            j = np.where(oids == pid)[0]
            self.marker[other].set_data(
                ([ox[j[0]]], [oy[j[0]]]) if len(j) else ([], []))
            res = self._fit(eid, iid)
            for f in ("extra", "intra"):
                data, model, resid = res[f]
                self._show_stamp(f, "data", data)
                self._show_stamp(f, "model", model)
                self._show_stamp(f, "residual", resid)
            print(f"\n=== pair  extra={eid}  intra={iid}  "
                  f"used={bool(row['used'])} ===")
            print(f"Noll {self.noll} (nm):",
                  np.round(res["zk"] * 1e3, 1))
        self.fig.canvas.draw_idle()


inspector = DonutPairInspector(inspect_visit or visits[0], inspect_raft)
